#### Demo Self Attention with visuals

In [1]:
import gradio as gr
import torch
import torch.nn.functional as F
import math
from transformers import AutoTokenizer
import numpy as np
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import io
import base64
import matplotlib.pyplot as plt

# Configurations
EMBED_DIM = 8  # Reduced dimension for simplicity in visualization

# Load a lightweight tokenizer to get realistic token blocks
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def compute_self_attention_results(sentence, target_index):
    """
    This function computes the self-attention weights for a given input sentence and a selected target token index.
    param sentence: str - The input sentence to analyze.
    param target_index: int - The index of the token to analyze as the Query in self-attention.
    return: tuple - A dictionary of attention weights, an HTML visualization string, and a step-by-step math log string.
    """
    # Fallback for empty inputs
    if not sentence.strip():
        return "Please enter a valid sentence.", {}, "", ""
    
    # 1. Tokenize text
    tokens = tokenizer.tokenize(sentence)
    num_tokens = len(tokens)
    
    if num_tokens == 0:
        return "No tokens found.", {}, "", ""
    
    # Bound-check the target index (Gradio dropdown returns integer index)
    target_index = int(target_index)
    if target_index >= num_tokens:
        target_index = num_tokens - 1

    # 2. Simulate Embeddings (Deterministic mock weights based on token IDs for repeatability)
    input_ids = tokenizer.convert_tokens_to_ids(tokens)
    embed_dim = EMBED_DIM
    
    # Create fixed mock embedding matrix using token IDs
    torch.manual_seed(42)
    embedding_layer = torch.nn.Embedding(tokenizer.vocab_size, embed_dim)
    embeddings = embedding_layer(torch.tensor([input_ids])) # Shape: [1, num_tokens, embed_dim]
    
    # 3. Project to Q, K, V
    # Deterministic projections
    W_q = torch.nn.Linear(embed_dim, embed_dim, bias=False)
    W_k = torch.nn.Linear(embed_dim, embed_dim, bias=False)
    W_v = torch.nn.Linear(embed_dim, embed_dim, bias=False)
    
    Q = W_q(embeddings) # [1, num_tokens, embed_dim]
    K = W_k(embeddings) # [1, num_tokens, embed_dim]
    V = W_v(embeddings) # [1, num_tokens, embed_dim]
    
    # Extract only the Query vector for our selected target token
    # Shape: [1, 1, embed_dim]
    Q_target = Q[:, target_index:target_index+1, :] 
    
    # 4. Calculate Attention Scores
    # Compute dot product: [1, 1, embed_dim] x [1, embed_dim, num_tokens] -> [1, 1, num_tokens]
    K_t = K.transpose(-2, -1)
    raw_scores = torch.matmul(Q_target, K_t).squeeze(0).squeeze(0) # Shape: [num_tokens]
    
    # 5. Scale Scores
    scaled_scores = raw_scores / math.sqrt(embed_dim)
    
    # 6. Softmax Normalization (Attention Weights)
    attention_weights = F.softmax(scaled_scores, dim=-1).tolist()
    
    return attention_weights, tokens, raw_scores, scaled_scores, embed_dim
    

def run_self_attention(sentence, target_index):
    """
    This function is the main handler for the Gradio interface. It computes the self-attention weights and formats the outputs for visualization.
    param sentence: str - The input sentence to analyze.
    param target_index: int - The index of the token to analyze as the Query in self-attention.
    return: tuple - A dictionary of attention weights for the Label component, an HTML"""  
    attention_weights, tokens, raw_scores, scaled_scores, embed_dim = compute_self_attention_results(
        sentence, 
        target_index
        )
    
    # --- Formatting the Outputs ---
    # Output A: Dictionary format for Gradio Label component (visual bars)
    num_tokens = len(tokens)
    
    weight_dict = {tokens[i]: attention_weights[i] for i in range(num_tokens)}
    
    # Output B: Step-by-Step Mathematical Explanation log
    math_log = (
        f"Selected Target Token (Query): '{tokens[target_index]}'\n"
        f"Embedding Dimension (d_k): {embed_dim}\n\n"
        f"Step 1 (Raw Vector Dot-Products):\n"
        f"Scores: {[round(x, 3) for x in raw_scores.tolist()]}\n\n"
        f"Step 2 (Scale by 1/√d_k):\n"
        f"Scaled: {[round(x, 3) for x in scaled_scores.tolist()]}\n\n"
        f"Step 3 (Softmax Exponential Regularisation):\n"
        f"Final Weights: {[round(x, 4) for x in attention_weights]}\n"
        f"Sum of Weights check: {round(sum(attention_weights), 4)}"
    )
    
    # Output C: HTML visualization emphasizing the weights inside text block
    html_output = "<div style='font-size: 1.2rem; line-height: 2rem;'>"
    for i, token in enumerate(tokens):
        weight = attention_weights[i]
        # Highlight intensity scales up to a visible soft background fill
        bg_opacity = min(weight * 1.5, 0.8) 
        is_target = "border: 2px solid #FF5733;" if i == target_index else ""
        
        html_output += (
            f"<span style='background-color: rgba(59, 130, 246, {bg_opacity:.2f}); "
            f"padding: 4px 8px; margin: 4px; border-radius: 4px; display: inline-block; {is_target}'>"
            f"<b>{token}</b> <small style='opacity: 0.7;'>({weight:.2%})</small>"
            f"</span>"
        )
    html_output += "</div>"
    
    return weight_dict, html_output, math_log

def create_attention_heatmap(sentence, target_index):
    """
    Generate a heatmap visualization of attention weights
    param sentence: str - The input sentence to analyze.
    param target_index: int - The index of the token to analyze as the Query in self-attention.
    return: str - An HTML string containing the heatmap image encoded in base64 for display in Gradio's HTML component.
    """
        
    attention_weights, tokens, raw_scores, scaled_scores, embed_dim = compute_self_attention_results(
        sentence, 
        target_index
        )
    num_tokens = len(tokens)
    
    # Create heatmap
    fig, ax = plt.subplots(figsize=(12, 2))
    
    # Reshape weights for visualization: [1, num_tokens]
    attention_weights = np.array(attention_weights)
    heatmap_data = attention_weights.reshape(1, -1)
    
    im = ax.imshow(heatmap_data, cmap='Blues', aspect='auto', vmin=0, vmax=max(attention_weights))
    
    # Set x-axis ticks to token positions
    ax.set_xticks(range(num_tokens))
    ax.set_xticklabels(tokens, rotation=45, ha='right')
    ax.set_yticks([])
    
    # Highlight target token
    ax.axvline(x=target_index, color='red', linewidth=3, linestyle='--', label=f"Target: '{tokens[target_index]}'")
    ax.legend(loc='upper right')
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax, orientation='vertical', pad=0.02)
    cbar.set_label('Attention Weight', rotation=270, labelpad=15)
    
    # Add text annotations with weight values
    for i, weight in enumerate(attention_weights):
        ax.text(i, 0, f'{weight:.2%}', ha='center', va='center', color='white' if weight > 0.5 else 'black', fontweight='bold')
    
    ax.set_title(f"Attention Weights for Token '{tokens[target_index]}'", fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    
    # Convert plot to base64 for HTML display
    buffer = io.BytesIO()
    plt.savefig(buffer, format='png', dpi=100, bbox_inches='tight')
    buffer.seek(0)
    image_base64 = base64.b64encode(buffer.read()).decode()
    plt.close()
    
    return f"<img src='data:image/png;base64,{image_base64}' style='width:100%;'>"


# Dynamic Dropdown update handler when input text changes
def update_dropdown(sentence):
    tokens = tokenizer.tokenize(sentence)
    choices = [(f"[{i}] {token}", i) for i, token in enumerate(tokens)]
    if not choices:
        return gr.Dropdown(choices=[], value=None, label="Choose Target Token")
    return gr.Dropdown(choices=choices, value=0, label="Choose Target Token (Query)")




# --- Gradio UI Layout Block ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🧠 Interactive Self-Attention Simulator")
    gr.Markdown(
        "Input a sentence to view tokenized blocks. "
        "Select your target token to evaluate how heavily it attends to every other token."
    )
    
    with gr.Row():
        with gr.Column(scale=1):
            input_text = gr.Textbox(
                value="Transformers handle context beautifully.", 
                label="Enter Sentence", 
                placeholder="Type something..."
            )
            target_token_dropdown = gr.Dropdown(
                choices=[("[0] transformers", 0), ("[1] handle", 1), ("[2] context", 2), ("[3] beautifully", 3), ("[4] .", 4)], 
                value=0, 
                label="Choose Target Token (Query)"
            )
            submit_btn = gr.Button("Calculate Attention Weights", variant="primary")
            
        with gr.Column(scale=1):
            gr.Markdown("### 📊 Attention Weight Distribution")
            weight_bars = gr.Label(label="Attention Scores")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 🎨 Text Heatmap Visualization (Target outline in Orange)")
            heatmap_html = gr.HTML()
            
    with gr.Row():
        with gr.Column():
            gr.Markdown("### 🧮 Step-by-Step Math Execution Log")
            math_execution_box = gr.Textbox(interactive=False, lines=10, label="Matrix Step Logging")
            
    with gr.Row():
        with gr.Column():
            gr.Markdown("### 🔥 Attention Heatmap")
            heatmap_viz = gr.HTML()

    # Wire up the automated dropdown generation
    input_text.change(fn=update_dropdown, inputs=input_text, outputs=target_token_dropdown)
    
    # Wire up calculation button
    # submit_btn.click(
    #     fn=run_self_attention, 
    #     inputs=[input_text, target_token_dropdown], 
    #     outputs=[weight_bars, heatmap_html, math_execution_box]
    # )
    submit_btn.click(
        fn=lambda sent, idx: (run_self_attention(sent, idx)[0], run_self_attention(sent, idx)[1], run_self_attention(sent, idx)[2], create_attention_heatmap(sent, idx)),
        inputs=[input_text, target_token_dropdown],
        outputs=[weight_bars, heatmap_html, math_execution_box, heatmap_viz]
    )

# if __name__ == "__main__":
#     demo.launch()


c:\Users\adeid\miniconda3\envs\ai-thursdays-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\adeid\AppData\Local\Temp\ipykernel_24508\2159858550.py:193: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


In [2]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


#### Provide a Gradio UI demo of a heatmap attension weight matrix for all the tokens in a sentence selected by the user

In [3]:
def create_attention_heatmap(sentence):
    """
    Overwritten: Generate a full self-attention matrix heatmap for all tokens in the sentence.
    Returns an HTML <img> with a base64-encoded PNG.
    """
    tokens = tokenizer.tokenize(sentence)
    num_tokens = len(tokens)
    if num_tokens == 0:
        return "No tokens to visualize."
    
    # Recreate deterministic embeddings and projections (match compute_self_attention_results seed)
    torch.manual_seed(42)
    embedding_layer = torch.nn.Embedding(tokenizer.vocab_size, EMBED_DIM)
    input_ids = tokenizer.convert_tokens_to_ids(tokens)
    embeddings = embedding_layer(torch.tensor([input_ids]))  # [1, T, D]
    
    W_q = torch.nn.Linear(EMBED_DIM, EMBED_DIM, bias=False)
    W_k = torch.nn.Linear(EMBED_DIM, EMBED_DIM, bias=False)
    W_v = torch.nn.Linear(EMBED_DIM, EMBED_DIM, bias=False)
    
    Q = W_q(embeddings)  # [1, T, D]
    K = W_k(embeddings)  # [1, T, D]
    
    # Compute full raw score matrix: [T, T]
    K_t = K.transpose(-2, -1)  # [1, D, T]
    raw_scores = torch.matmul(Q, K_t).squeeze(0)  # [T, T]
    scaled_scores = raw_scores / math.sqrt(EMBED_DIM)
    attention_matrix = F.softmax(scaled_scores, dim=-1).detach().numpy()  # [T, T]
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=(max(6, num_tokens*0.6), max(4, num_tokens*0.45)))
    im = ax.imshow(attention_matrix, cmap="Blues", aspect="auto", vmin=0, vmax=attention_matrix.max())
    
    # Ticks and labels
    ax.set_xticks(range(num_tokens))
    ax.set_yticks(range(num_tokens))
    ax.set_xticklabels(tokens, rotation=45, ha="right")
    ax.set_yticklabels(tokens)
    
    ax.set_xlabel("Key tokens")
    ax.set_ylabel("Query tokens")
    ax.set_title("Full Self-Attention Weight Matrix")
    
    # Annotate values
    for i in range(num_tokens):
        for j in range(num_tokens):
            val = attention_matrix[i, j]
            color = "white" if val > 0.5 * attention_matrix.max() else "black"
            ax.text(j, i, f"{val:.2%}", ha="center", va="center", color=color, fontsize=8)
    
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Attention Weight", rotation=270, labelpad=15)
    plt.tight_layout()
    
    # Convert to base64 HTML image
    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=100, bbox_inches="tight")
    buf.seek(0)
    img_b64 = base64.b64encode(buf.read()).decode()
    plt.close(fig)
    
    return f"<img src='data:image/png;base64,{img_b64}' style='width:100%;'>"

##### Provide a simple Gradio UI to enable the user to provide the target sentence in a textbox and click a button to invoke the create_attention_heatmap function with the selected sentence

In [4]:
with gr.Blocks(theme=gr.themes.Soft()) as heatmap_demo:
    gr.Markdown("## 🔥 Full Self-Attention Matrix Heatmap Demo")
    sentence_input = gr.Textbox(
        label="Enter Sentence",
        placeholder="Type a sentence to visualize full attention weights...",
        value="The quick brown fox jumps over the lazy dog.",
        lines=2,
    )
    generate_button = gr.Button("Generate Attention Heatmap")
    heatmap_output = gr.HTML()

    generate_button.click(
        fn=create_attention_heatmap,
        inputs=sentence_input,
        outputs=heatmap_output,
    )

heatmap_demo.launch()

C:\Users\adeid\AppData\Local\Temp\ipykernel_24508\2337731515.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as heatmap_demo:


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
